# 05 — Model Finalization, SHAP Explainability & Production Export

## Objective
This notebook documents the final production pipeline for the **Disaster Type Prediction** project.
It trains the best hyperparameter-tuned **Random Forest Pipeline (with SMOTETomek resampling)**, evaluates its test performance, verifies **Explainable AI (SHAP)** integration, and exports all artifacts required for the Interactive Model Auditor Web Application.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import shap

### 1. Load Processed Feature-Selected Dataset

In [2]:
data_path = "../data/processed/disaster_processed_feature_selected.csv"
data = pd.read_csv(data_path)

X = data.drop(columns=["disaster_type_encoded"])
y = data["disaster_type_encoded"]

print(f"Dataset shape: {data.shape}")
print(f"Features count: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")

### 2. Train-Test Stratified Split (80/20)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

CLASS_NAMES = [
    "Drought",
    "Earthquake",
    "Epidemic",
    "Extreme Temperature",
    "Flood",
    "Landslide",
    "Storm",
    "Wildfire"
]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

### 3. Build & Fit Best Tuned Random Forest Pipeline

In [4]:
rf_pipeline = Pipeline([
    ("smotetomek", SMOTETomek(random_state=42)),
    ("random_forest", RandomForestClassifier(
        n_estimators=100,
        min_samples_split=5,
        min_samples_leaf=1,
        max_features="log2",
        max_depth=40,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ))
])

print("Fitting best RF Pipeline...")
rf_pipeline.fit(X_train, y_train)
print("Training complete!")

### 4. Evaluate on Test Set

In [5]:
y_pred = rf_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1 Macro: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

### 5. Verify SHAP Explainability (TreeExplainer)

In [6]:
rf_model = rf_pipeline.named_steps["random_forest"]
explainer = shap.TreeExplainer(rf_model)

# Test explanation on first test sample
sample_x = X_test.iloc[[0]]
sample_shap = explainer.shap_values(sample_x)
pred_class = rf_pipeline.predict(sample_x)[0]

print(f"Sample predicted as: {CLASS_NAMES[pred_class]}")
print("SHAP values calculated successfully.")

### 6. Export Artifacts for Web Application

In [7]:
joblib.dump(rf_pipeline, "../models/rf_pipeline.joblib")
print("Exported: ../models/rf_pipeline.joblib")